# Разведочный анализ данных (EDA) агрегата `UNIT`

**Зачем.** Прежде чем искать аномалии, нужно понять, что такое *норма* для этого котла и какие данные ей годятся: когда котёл реально работает, каким датчикам можно доверять, от чего зависят показания, стабильна ли норма во времени и что было видно перед ремонтами.

| Раздел | Вопрос |
|---|---|
| 1 | Что есть в данных |
| 2 | Какие были ремонты: аварийный, плановый, резерв (по типу заявки) |
| 3 | Когда котёл реально работает и сколько длится пуск |
| 3.1 | Совпадает ли время заявок с остановами по данным (часовой пояс) |
| 3.2 | Сколько часов нормальной работы остаётся моделям и куда уходят остальные; какие условия найдёт ноутбук 02 |
| 4 | Каким датчикам можно доверять |
| 5 | Насколько показания определяются нагрузкой и состоянием механизмов |
| 6 | Стабильна ли норма во времени (дрейф) |
| 7 | Почему пороги берутся «по факту», а не по правилу 3σ |
| 8 | Дубли датчиков и «память» отклонений |
| 9 | Текстовые датчики: задвижки, механизмы, защиты |
| 10 | Какие датчики отклонялись перед ремонтами |
| 10.1 | Небаланс вода − пар: эпизоды и что было после них |
| 11 | Итоги и решения для ноутбука моделей |

Результаты сохраняются в `eda_results/<агрегат>/` для просмотра. Версия 4.1 ноутбука 02 не использует статистики EDA всего периода для обучения и не переносит из них исключения автоматически. Рядом должен лежать `k2_lib.py`.

`USE_SENSOR_LIST = True` — смотреть только датчики из Excel-списка `SENSOR_LIST_FILE` (лист на котёл, столбцы `Kod`, `col`, `агрегат`), как в 02. Шаблоны имён датчиков (условия, расход питательной воды и пара, подозрительные значения) — в `k2_lib.py`, блок «ШАБЛОНЫ ИМЁН ДАТЧИКОВ»; для одного агрегата их можно заменить параметрами `RX_…`.

Статистики EDA ретроспективные. Нормативы, ступенчатость и постоянство — поводы для проверки, а не доказательство неисправности или причина исключать датчик. Заявка задаёт момент останова; время по телеметрии его не переопределяет.


## 0. Параметры

In [ ]:
# Рабочая папка и локальные библиотеки; установки пакетов нет.
from pathlib import Path
import os, sys
_notebook_parent = Path(globals().get('__file__', '01_EDA.ipynb')).resolve().parent
_candidates = [_notebook_parent, Path.cwd() / 'legacy', Path.cwd().parent / 'legacy']
PROJECT_DIR = next((p for p in _candidates if (p / 'lib' / 'legacy_group.py').is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Откройте ноутбук из папки legacy: рядом должна быть папка lib.')
PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
for _path in (PROJECT_DIR, PROJECT_DIR / 'lib'):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
__file__ = str(PROJECT_DIR / '01_EDA.ipynb')

UNIT = 'К-2'                                    # агрегат: пути ниже — от него (opc_to_wide.py пишет в opc_wide/<агрегат>/)
HOURLY_PATH = f'opc_wide/{UNIT}/hourly.parquet'  # часовая таблица (opc_to_wide.py)
REPAIRS_PATH = f'opc_wide/{UNIT}/repairs.csv'   # заявки агрегата с типом и категорией (opc_to_wide.py)
RUN_COL = None                                  # датчик «работает / стоит»; None — «Расход перегретого пара» / «F перегретого пара»
                                                # (шаблон RX_RUN в k2_lib.py); не нашёлся — ошибка и кандидаты по данным (раздел 3)
RUN_FRAC = 0.25                                 # работает, если он > RUN_FRAC × медиана в работе
MAX_MISSING = 0.3                               # больше пропусков при работе — датчик не годится
MIN_NORMAL_H = 720                              # проверка «как в эксплуатации» начинается с месяца, перед которым набралось
                                                # столько часов нормальной работы (720 — месяц); решение — first_test
PRE_HOURS = 72                                  # сколько часов работы смотреть перед ремонтами (раздел 10)
OUT_DIR = f'eda_results/{UNIT}'                 # решения EDA агрегата — их подхватывает ноутбук 02
USE_SENSOR_LIST = False                         # True — только датчики из Excel-списка SENSOR_LIST_FILE (датчик работы — всегда)
SENSOR_LIST_FILE = 'sensor_roles.xlsx'               # лист на котёл («КА-1», «КА-2» …), столбцы Kod (код OPC), col (имя), агрегат (К-1 …)
EXCL_FAIL_D = 14                                # как в 02 (PRE_WINDOW_D + NEUTRAL_D): столько суток перед аварией — не норма
AFTER_REPAIR_H = 48                             # как в 02: столько часов после ремонта или резерва — не норма
TRAIN_HOURS, MIN_TRAIN_H = 6500, 500            # как в 02: окно обучения (ч нормы) и меньше скольких часов модель месяца не строится
IMB_PCT, IMB_HOURS = 3.0, 6                     # раздел 10.1: небаланс вода − пар больше IMB_PCT % нагрузки дольше IMB_HOURS ч — эпизод
# шаблоны имён датчиков — в k2_lib.py (блок «ШАБЛОНЫ ИМЁН ДАТЧИКОВ»); None — оттуда, свои — для этого агрегата; {N} — номер агрегата
RX_CONDITIONS = None                            # условия, которые найдёт ноутбук 02: холодный воздух, температура питательной воды
RX_FEED_WATER_FLOW = None                       # раздел 10.1: расход питательной воды
RX_STEAM_FLOW = None                            # раздел 10.1: расход пара (если датчик работы — не расход пара)

MANUAL_EXCLUDE_SENSORS = []  # только явно выбранные полные имена; по умолчанию никто не исключается


In [ ]:
import os, re, json, warnings, importlib
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy import stats
from scipy.cluster import hierarchy
from IPython.display import display

import k2_lib as K
K = importlib.reload(K)                     # всегда свежая версия k2_lib.py
if not hasattr(K, 'Prep'):
    raise ImportError(f'Рядом с ноутбуком старая k2_lib.py ({K.__file__}) — замените её новой версией.')
from k2_lib import say, md, short, fmt_list, pct, STATE_COL, ST_WORK, ST_FAIL, ST_PLAN, ST_NODATA, EV_FAIL, EV_PLAN, EV_RES, M_STOP, M_START, M_RUN
K.setup_plots()
os.makedirs(OUT_DIR, exist_ok=True)
EV_COL = {EV_FAIL: '#e74c3c', EV_PLAN: '#3498db', EV_RES: '#f39c12'}
EV_NAME = {EV_FAIL: 'авария', EV_PLAN: 'плановый', EV_RES: 'резерв'}
def lab(c, n=48):
    """Подпись датчика: длинное имя сокращается в середине, конец (номер точки, сторона) сохраняется."""
    c = short(c, 200)
    return c if len(c) <= n else c[:n - 19] + '…' + c[-18:]

## 1. Данные
**Часовая таблица** (от `opc_to_wide.py`): числа — среднее за час с весами по времени, тексты — состояние, которое держалось в часе дольше всего. Столбец `Состояние` — метка по типу и категории заявки: «Работа» (заявки нет), «Плановый ремонт», «Аварийный ремонт», «Резерв», «Нет данных» (вне периода файла заявок).

In [ ]:
df, types, load_info = K.load_hourly(HOURLY_PATH, '1h')
num_all, txt_cols = types['num'], types['txt']
RX_CONDITIONS = K.unit_rx(RX_CONDITIONS or K.RX_CONDITIONS, UNIT)        # шаблоны имён: свои или из k2_lib, {N} -> номер
RX_FEED_WATER_FLOW = K.unit_rx(RX_FEED_WATER_FLOW or K.RX_FEED_WATER_FLOW, UNIT)
RX_STEAM_FLOW = K.unit_rx(RX_STEAM_FLOW or K.RX_STEAM_FLOW, UNIT)
SENSOR_LIST_INFO = None
if USE_SENSOR_LIST:                              # только датчики из Excel-списка; датчик работы котла ищется среди всех
    NUM_FULL = list(num_all)
    keep_, SENSOR_LIST_INFO = K.sensor_list(SENSOR_LIST_FILE, UNIT, num_all + txt_cols,
                                            os.path.join(os.path.dirname(HOURLY_PATH), 'sensors.csv'))
    keep_ = set(keep_)
    num_all = [c for c in num_all if c in keep_]
    txt_cols = [c for c in txt_cols if c in keep_]
state = df[STATE_COL].map(lambda v: v.strip() if isinstance(v, str) and v.strip() else None).ffill(limit=3).fillna(ST_NODATA)
days = (df.index[-1] - df.index[0]) / pd.Timedelta('1D')
lab_cnt = state.value_counts()
say(f'Период: **{df.index[0]:%Y-%m-%d} — {df.index[-1]:%Y-%m-%d}** ({days:.0f} сут, {len(df):,} часов).',
    f'Датчиков: **{len(num_all)}** числовых, **{len(txt_cols)}** текстовых (задвижки, механизмы, защиты); пустых столбцов: {len(types["empty"])}.',
    'Метки по заявкам: ' + ', '.join(f'{k} — {v:,} ч' for k, v in lab_cnt.items()) + '.',
    (f'**Только датчики из списка** `{SENSOR_LIST_FILE}`: в списке {SENSOR_LIST_INFO["listed"]}, найдено в данных '
     f'{SENSOR_LIST_INFO["found"]}' + (f'; нет в данных: {fmt_list([short(c, 35) for c in SENSOR_LIST_INFO["missing"]], 8)}'
                                       if SENSOR_LIST_INFO['missing'] else '') + '.') if SENSOR_LIST_INFO else '')

## 2. Ремонты: аварийный, плановый, резерв
Метка ставится конвертером по **типу и категории** заявки:

| Тип | Категория | Метка |
|---|---|---|
| ХР, ВПр, ЗРР, ИСП (холодный резерв, вынужденный простой, заявленный режим, испытания) | любая | Резерв |
| остальные (АР, СР, ТР, КР, …) | Плановый | Плановый ремонт |
| остальные (АР, СР, ТР, КР, …) | любая другая | Аварийный ремонт |

Резервы в проверке моделей — контроль: котёл перед ними исправен, тревог быть не должно.

In [ ]:
events = K.repair_events(state, 0)
rep = K.load_repairs(REPAIRS_PATH) if REPAIRS_PATH and os.path.exists(REPAIRS_PATH) else pd.DataFrame()
if len(rep):
    from boiler_monitoring import read_ticket_events
    exact_events = read_ticket_events(REPAIRS_PATH, df.index[-1] + pd.Timedelta('1h'))
    events = exact_events.rename(columns={'type': 'state'})
    events['type'] = events.state.map({ST_FAIL: EV_FAIL, ST_PLAN: EV_PLAN, 'Резерв': EV_RES})
    events['hours'] = (events.end - events.start) / pd.Timedelta('1h')
    events['n_ep'] = 1

test_h = df['_Тип'].astype(str).isin(['ИСП', 'ЗРР']) if '_Тип' in df else pd.Series(False, index=df.index)   # испытания, заявленный режим
if len(rep):
    used = rep[rep['учтена']]
    display(used.groupby(['тип', 'категория', 'состояние'], dropna=False).size().rename('заявок').reset_index())
rows = []
for i, e in events.iterrows():
    prev_end = events.end[events.end <= e.start].max() if (events.end <= e.start).any() else df.index[0]
    kinds = txt = '—'
    if len(rep):
        ov = used[(used['начало'] < e.end - pd.Timedelta('1h')) & (used['окончание'].fillna(e.end) > e.start)]   # метка часа — его конец
        kinds = fmt_list(sorted({f'{t} / {c}' for t, c in zip(ov['тип'].astype(str), ov['категория'].fillna('—').astype(str))}), 3)
        txt = fmt_list(ov['содержание'].dropna().astype(str).str.slice(0, 70).unique(), 2)
    rows.append({'тип': EV_NAME[e.type], 'начало': e.start, 'конец': e.end, 'длительность, сут': round(e.hours / 24, 1),
                 'работы перед ним, сут': round((e.start - prev_end) / pd.Timedelta('1D'), 1), 'заявки: тип / категория': kinds,
                 'содержание заявок': txt})
display(pd.DataFrame(rows))
# все аварии — одним файлом на проверку экспертам: настоящая ли это авария (а не плановые работы под аварийной заявкой)
os.makedirs(OUT_DIR, exist_ok=True)
fail_rev = pd.DataFrame([r for r in rows if r['тип'] == EV_NAME[EV_FAIL]])
if len(fail_rev):
    fail_rev = fail_rev.drop(columns='тип').assign(**{'авария? (да / нет)': '', 'что повреждено, причина': ''})
    fail_rev.to_csv(os.path.join(OUT_DIR, 'failures_review.csv'), sep=';', index=False, encoding='utf-8-sig')
fails, plans, reserves = events[events.type.eq(EV_FAIL)], events[events.type.eq(EV_PLAN)], events[events.type.eq(EV_RES)]
skipped = rep[~rep['учтена']] if len(rep) else rep
say((f'Аварии на проверку экспертам — `{OUT_DIR}/failures_review.csv`: по каждой дата, длительность, тип и категория заявки, '
     'содержание работ и пустые столбцы «авария? (да / нет)» и «что повреждено, причина». Аварийная заявка не всегда авария '
     '(бывают плановые работы и модернизация под внеплановой категорией) — такие стоит отметить «нет».') if len(fail_rev) else '',
    f'Ремонтов и резервов: **{len(events)}** — аварий **{len(fails)}** ({fmt_list([f"{t:%Y-%m-%d}" for t in fails.start])}), '
    f'плановых {len(plans)}, выводов в резерв {len(reserves)}' + (f' ({fmt_list([f"{t:%Y-%m-%d}" for t in reserves.start])})' if len(reserves) else '') + '.',
    f'Испытания и заявленный режим (ИСП, ЗРР): {int(test_h.sum())} ч — котёл работает, но это не обычный режим: исключены из всего (норма, фон, окна перед авариями).' if test_h.any() else '',
    (f'Не учтено заявок: {len(skipped)}' + (' (' + ', '.join(f'{k} — {v}' for k, v in skipped['примечание'].value_counts().items()) + ')'
                                            if 'примечание' in skipped else '') + '.') if len(skipped) else '')   # «примечание» — только у opc_to_wide.py

## 3. Когда котёл реально работает
Метка «Работа» значит лишь, что нет заявки: котёл при этом может стоять (резерв, простой). Поэтому «работает» определяется по данным — по датчику работы — **расход перегретого пара** («Расход перегретого пара» или «F перегретого пара», шаблон `RX_RUN` в `k2_lib.py`; его можно задать и явно параметром `RUN_COL`). Он проверяется по данным: в размеченной «Работе» должен быть высоким, в ремонтах и резервах — около нуля; если это не так — предупреждение и кандидаты. Котёл работает, если датчик больше `RUN_FRAC` × медиана в размеченной работе. После включения котёл несколько часов выходит на режим (**пуск**): доля датчиков вне обычного диапазона высокая. Длительность пуска находим по данным: когда эта доля опускается до уровня установившейся работы.

In [ ]:
RUN_CANDIDATES = NUM_FULL if USE_SENSOR_LIST else num_all     # датчик работы нужен всегда, даже если его нет в списке
try:
    run_col, run_src = K.run_column(RUN_CANDIDATES, RUN_COL), ('RUN_COL' if RUN_COL else 'по имени — расход перегретого пара')
except KeyError as err:                                 # нет «Расход перегретого пара»: кандидаты по данным — для RUN_COL
    _, run_tab = K.run_column_by_data(df, RUN_CANDIDATES, state)
    if run_tab is not None and len(run_tab):
        md('**Датчик работы котла не найден по имени** («Расход перегретого пара» / «F перегретого пара»). Кандидаты по данным '
           '(высокий в «Работе», около нуля в ремонтах и резервах) — подходящий задайте в RUN_COL:')
        display(K.fmt_df(run_tab.head(8).set_index('датчик'), {'в «Работе» выше 25% медианы': '{:.0%}', 'в ремонтах и резервах около нуля': '{:.0%}',
                                                               'медиана в «Работе»': '{:.4g}'}))
    raise KeyError(f'{err} Кандидаты по данным — в таблице выше.') from None
if not RUN_COL:                                         # проверка по данным (только предупреждение): высокий в «Работе»,
    best_d, tab_d = K.run_column_by_data(df, RUN_CANDIDATES, state)   # около нуля в ремонтах и резервах
    sc_d = tab_d.set_index('датчик')['оценка'] if len(tab_d) else pd.Series(dtype=float)
    if run_col in sc_d.index and sc_d[run_col] < 0.8:
        md(f'**ВНИМАНИЕ: «{short(run_col, 50)}» плохо отличает работу от останова** (оценка {sc_d[run_col]:.2f}, нужно ≥ 0.8) — '
           'проверьте датчик (пропуски, залипание, другой масштаб); '
           + (f'по данным лучше всех «{short(best_d, 50)}» — если согласны, задайте RUN_COL. ' if best_d is not None and best_d != run_col
              else 'при необходимости задайте RUN_COL. ') + 'Кандидаты:')
        display(K.fmt_df(tab_d.head(8).set_index('датчик'), {'в «Работе» выше 25% медианы': '{:.0%}',
                                                              'в ремонтах и резервах около нуля': '{:.0%}', 'медиана в «Работе»': '{:.4g}'}))
x_run = df[run_col].astype(float)
run_thr = K.run_threshold(x_run, state, RUN_FRAC)
mode0, since0, _ = K.detect_mode(x_run, run_thr, startup_h=0, prestop_h=0, min_stop_h=2)
running = mode0.ne(M_STOP)

# длительность пуска: доля датчиков вне 1–99% установившейся работы по часам после включения
ref = running & (since0 >= 72) & state.eq(ST_WORK)
cand = [c for c in num_all if df.loc[ref, c].notna().mean() > 0.7 and df.loc[ref, c].nunique() > 5]
lo_, hi_ = df.loc[ref, cand].quantile(0.01), df.loc[ref, cand].quantile(0.99)
out_share = ((df[cand] < lo_) | (df[cand] > hi_)).astype(float).where(df[cand].notna()).mean(axis=1)
base = max(out_share[ref].quantile(0.95), 1.5 * out_share[ref].median())
run_ep = K.episodes(running).reset_index(drop=True)
H1, curves = 96, []
for i, r in run_ep.iterrows():
    if r.value and i > 0 and run_ep.loc[i - 1, 'hours'] >= 12:
        seg = out_share.loc[r.start:r.end - pd.Timedelta('1h')].iloc[:H1].to_numpy()
        curves.append(np.r_[seg, np.full(H1 - len(seg), np.nan)])
C_ = np.array(curves) if curves else np.full((1, H1), np.nan)
med_c = np.nanmedian(C_, 0)
ok = med_c <= base
startup_h = int(np.clip(next((h for h in range(H1 - 12) if np.all(ok[h:h + 12])), H1), 6, 72))
mode, _, _ = K.detect_mode(x_run, run_thr, startup_h, 3, 2)
running = mode.ne(M_STOP)
steady = mode.eq(M_RUN) & state.eq(ST_WORK) & ~test_h

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
hi = np.nanpercentile(x_run, 99.8)
bins = np.linspace(0, hi, 70)
axes[0].hist(x_run[running].clip(upper=hi).dropna(), bins=bins, color='#27ae60', alpha=.8, label='работает')
axes[0].hist(x_run[~running].clip(upper=hi).dropna(), bins=bins, color='#95a5a6', alpha=.8, label='стоит')
axes[0].axvline(run_thr, color='#c0392b', ls='--', lw=1.5, label=f'порог {run_thr:.0f}')
axes[0].set_yscale('log'); axes[0].set_xlabel(short(run_col, 40)); axes[0].set_ylabel('часов (лог. шкала)')
axes[0].set_title('Работает или стоит: два «горба», порог между ними'); axes[0].legend(loc='upper center')
h_ = np.arange(H1)
axes[1].fill_between(h_, np.nanpercentile(C_, 25, 0) * 100, np.nanpercentile(C_, 75, 0) * 100, color='#f5b041', alpha=.35, label='половина пусков')
axes[1].plot(h_, med_c * 100, color='#d35400', lw=2, label=f'медиана ({len(curves)} пусков)')
axes[1].axhline(base * 100, color='gray', ls='--', lw=1, label='уровень установившейся работы')
axes[1].axvline(startup_h, color='k', lw=1.5, label=f'конец пуска: {startup_h} ч')
axes[1].set_xlabel('часов после включения'); axes[1].set_ylabel('% датчиков вне обычного диапазона')
axes[1].set_title('Сколько длится пуск'); axes[1].legend(loc='upper right')
plt.show()

def shade(ax):
    """Остановы — серым, ремонты — цветом по типу."""
    st = K.episodes(running)
    for r in st[st.value.eq(False)].itertuples():
        ax.axvspan(r.start, r.end, color='#bdc3c7', alpha=.35, lw=0)
    for e in events.itertuples():
        ax.axvspan(e.start, e.end, color=EV_COL[e.type], alpha=.25, lw=0)
SHADE_HANDLES = [Patch(color=EV_COL[EV_FAIL], alpha=.4, label='аварийный ремонт'), Patch(color=EV_COL[EV_PLAN], alpha=.4, label='плановый ремонт'),
                 Patch(color=EV_COL[EV_RES], alpha=.4, label='резерв'), Patch(color='#bdc3c7', label='котёл стоит')]

fig, ax = plt.subplots(figsize=(15, 3.6))
shade(ax)
d = x_run.where(running).resample('1D').mean()
ax.plot(d.index, d, color='#2c3e50', lw=1)
ax.set_ylabel(short(run_col, 30)); K.date_axis(ax)
ax.set_title('Нагрузка котла (среднее за сутки работы), ремонты и остановы')
fig.legend(handles=[Line2D([0], [0], color='#2c3e50', lw=1.5, label='нагрузка')] + SHADE_HANDLES, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.12), fontsize=9)
plt.show()
stop_ep = K.episodes(running)
stop_ep = stop_ep[stop_ep.value.eq(False) & (stop_ep.start > df.index[0])]
no_req = sum(not ((events.start < r.end) & (events.end > r.start)).any() for r in stop_ep.itertuples())
say(f'Котёл работает, если {short(run_col, 40)} > **{run_thr:.0f}**. Остановов (≥ 2 ч): {len(stop_ep)}, из них без заявки: {no_req}.',
    f'При метке «Работа» котёл стоял {pct((state.eq(ST_WORK) & ~running).sum() / max(state.eq(ST_WORK).sum(), 1))} часов — это остановы без заявки (резерв, простой).',
    f'**Пуск** длится около **{startup_h} ч**: столько после включения доля датчиков вне обычного диапазона выше уровня установившейся работы. '
    'Эти часы не считаются нормой.',
    f'**Установившаяся работа** (котёл работает, не пуск, метка «Работа», не испытания): **{steady.sum():,} ч** — из неё берётся норма для моделей.')

### 3.1 Совпадает ли время заявок с остановами по данным
Если время в файле заявок записано в другом часовом поясе, метки ремонтов сдвинуты относительно данных на одно и то же
число часов. Тогда последние часы перед аварией оказываются «ремонтом», а часы после пуска — «работой». Проверка: для
каждого ремонта и резерва — начало по заявке против ближайшего останова котла по данным (± 2 сут) и конец по заявке против
пуска. Плановые ремонты и резервы обычно начинаются вместе с остановом; аварийную заявку часто оформляют позже останова.
Заявку обычно открывают **после** останова и закрывают **до** пуска. Если почти у всех событий заявка начинается
**раньше** останова (котёл ещё работает, а по метке уже ремонт) или заканчивается **позже** пуска — дело в часовом поясе:
поправка `REPAIRS_SHIFT_H` в `opc_to_wide.py` (новые таблицы) или `shift_labels.py` (уже собранные). Величину сдвига
ограничивают с двух сторон начало и конец ремонтов — поэтому она дана диапазоном.

In [ ]:
stop_t = stop_ep['start'] if len(stop_ep) else pd.Series(dtype='datetime64[ns]')     # остановы по данным (раздел 3)
start_ep = K.episodes(running)
start_t = start_ep.loc[start_ep.value.eq(True) & (start_ep.start > df.index[0]), 'start']  # пуски по данным
near_ = lambda ts, t: ts.iloc[int(np.argmin(np.abs((ts - t) / pd.Timedelta('1h'))))] if len(ts) else None
rows_off = []
for e in events.itertuples():
    s_ = near_(stop_t[(stop_t >= e.start - pd.Timedelta('2D')) & (stop_t <= e.start + pd.Timedelta('2D'))], e.start)
    b_ = near_(start_t[(start_t >= e.end - pd.Timedelta('2D')) & (start_t <= e.end + pd.Timedelta('2D'))], e.end)
    if s_ is None and b_ is None:
        continue
    rows_off.append({'тип': EV_NAME[e.type], 'начало по заявке': e.start, 'останов по данным': s_,
                     'заявка позже останова, ч': (e.start - s_) / pd.Timedelta('1h') if s_ is not None else np.nan,
                     'конец по заявке': e.end, 'пуск по данным': b_,
                     'заявка закрыта позже пуска, ч': (e.end - b_) / pd.Timedelta('1h') if b_ is not None else np.nan})
OFF = pd.DataFrame(rows_off)
LABEL_OFFSET_H = 0
verdict_off = 'Момент останова принят по началу заявки. Расхождения с телеметрией показаны только для просмотра; время заявок не сдвигается.'
if len(OFF):
    display(K.fmt_df(OFF, {'заявка позже останова, ч': '{:+.0f}', 'заявка закрыта позже пуска, ч': '{:+.0f}'}))
say(verdict_off)


### 3.2 Сколько часов нормальной работы остаётся моделям
Как в ноутбуке 02: модель каждого месяца учится только на **нормальной работе** до него — котёл работает, не пуск и не
последние 3 ч перед остановом, метка «Работа», не испытания, не 48 ч после ремонта, не 14 сут перед аварией. Таблица
показывает, какое правило сколько часов отсекает; ниже — нормальная работа по месяцам и месяцы, где модели не на чем
учиться. Последний пункт — условия работы, которые ноутбук 02 найдёт по именам (температура питательной воды и холодного
воздуха): если какого-то нет, проверьте имя датчика (шаблоны `RX_CONDITIONS` в `k2_lib.py`).

In [ ]:
after_rep = np.zeros(len(df), bool)
for e in events.itertuples():
    after_rep |= (df.index >= e.end) & (df.index < e.end + pd.Timedelta(hours=AFTER_REPAIR_H))
pre_fail = (K.days_to_next(df.index, fails.start) <= EXCL_FAIL_D) if len(fails) else np.zeros(len(df), bool)
pre_fail = np.asarray(pre_fail, bool)
steps_ = [('всего часов в таблице', np.ones(len(df), bool), ''),
          ('котёл работает', running.to_numpy(), f'датчик работы «{short(run_col, 30)}» и RUN_FRAC'),
          (f'не пуск ({startup_h} ч) и не 3 ч перед остановом', mode.eq(M_RUN).to_numpy(), 'длительность пуска — раздел 3'),
          ('метка заявок «Работа»', state.eq(ST_WORK).to_numpy(), 'ремонты, резервы; «Нет данных» — вне периода файла заявок'),
          ('не испытания и не 48 ч после ремонта', ~test_h.to_numpy() & ~after_rep, 'AFTER_REPAIR_H'),
          (f'не {EXCL_FAIL_D} сут перед аварией ({len(fails)} аварий)', ~pre_fail, 'PRE_WINDOW_D + NEUTRAL_D в 02')]
keep_, tab_n = np.ones(len(df), bool), []
for name_, m_, how_ in steps_:
    before_ = int(keep_.sum())
    keep_ = keep_ & m_
    tab_n.append({'шаг': name_, 'осталось часов': int(keep_.sum()), 'отсечено': before_ - int(keep_.sum()), 'чем настраивается': how_})
NORMAL02 = pd.Series(keep_, index=df.index)            # норма для обучения, как в 02 (без недель перед авариями)
display(pd.DataFrame(tab_n))
nodata_run = int((running & mode.eq(M_RUN) & state.eq(ST_NODATA)).sum())

per_m = pd.DataFrame({'норма для обучения': NORMAL02, 'работа, но не норма': running & ~NORMAL02}).groupby(df.index.to_period('M')).sum()
first_ = K.first_test_month(df.index, steady & ~pd.Series(after_rep, index=df.index), MIN_NORMAL_H)
short_m = []
if first_:
    for m0 in pd.date_range(first_, df.index[-1], freq='MS'):
        win_ = (df.index < m0) & (df.index >= m0 - pd.Timedelta(days=1095))
        if min(int((NORMAL02.to_numpy() & win_).sum()), TRAIN_HOURS) < MIN_TRAIN_H:
            short_m.append(f'{m0:%Y-%m}')
fig, ax = plt.subplots(figsize=(15, 3.6))
xm = [str(p_) for p_ in per_m.index]
ax.bar(xm, per_m['норма для обучения'], color='#27ae60', label='нормальная работа (идёт в обучение)')
ax.bar(xm, per_m['работа, но не норма'], bottom=per_m['норма для обучения'], color='#f5b041', label='котёл работает, но не норма')
if first_:
    ax.axvline(xm.index(f'{pd.Timestamp(first_):%Y-%m}') - 0.5 if f'{pd.Timestamp(first_):%Y-%m}' in xm else len(xm) - 0.5,
               color='k', ls='--', lw=1.2, label=f'начало проверки (≥ {MIN_NORMAL_H} ч нормы до него)')
ax.set_ylabel('часов'); ax.tick_params(axis='x', rotation=60); ax.legend(loc='upper left', fontsize=9)
ax.set_title('Часы нормальной работы по месяцам')
plt.show()

N_ = K.unit_number(UNIT)
cand_c = [c for c in num_all if c != run_col]
COND_FOUND = {}
for nm_, rx_ in RX_CONDITIONS.items():
    hits_ = [c for c in cand_c if K.name_hit(rx_, c)]
    COND_FOUND[nm_] = min(hits_, key=lambda c: df.loc[NORMAL02, c].isna().mean()) if hits_ else None
big_ = sorted(tab_n[1:], key=lambda r: -r['отсечено'])[:3]
say(f'Нормальной работы для обучения: **{int(NORMAL02.sum()):,} ч** из {len(df):,} часов в таблице. Больше всего отсекают: '
    + '; '.join(f'«{r["шаг"]}» — {r["отсечено"]:,} ч' for r in big_) + '.',
    (f'Котёл работал, а метка «Нет данных» (вне периода файла заявок): {nodata_run:,} ч — если файл заявок короче данных, '
     'эти часы теряются для нормы (в новом проекте — `labels.nodata_as_work: true`).') if nodata_run > 24 else '',
    (f'Проверка «как в эксплуатации» может начаться с **{pd.Timestamp(first_):%Y-%m}** (до него ≥ {MIN_NORMAL_H} ч нормы).'
     if first_ else f'**Нормальной работы меньше {MIN_NORMAL_H} ч** — проверять модели не на чем: смотрите таблицу выше.'),
    (f'Месяцы, где модели не на чем учиться (меньше {MIN_TRAIN_H} ч нормы до них): {fmt_list(short_m, 12)}.' if short_m else
     (f'Во всех месяцах с {pd.Timestamp(first_):%Y-%m} модели есть на чём учиться (≥ {MIN_TRAIN_H} ч).' if first_ else '')),
    'Условия для ноутбука 02: ' + '; '.join(f'{k} — «{short(v, 40)}»' if v else f'{k} — **не найдена** (проверьте имя; шаблоны RX_CONDITIONS в k2_lib.py)'
                                            for k, v in COND_FOUND.items()) + '.')

## 4. Каким датчикам можно доверять
Всё — только по часам, когда котёл работает (при останове датчики показывают ноль или мусор, это не их неисправность). Итог по каждому датчику:
- **мало данных** — пропусков больше `MAX_MISSING`;
- **константа** — одно значение ≥ 98% времени;
- **норматив/задание** — расчётные нормативы и уставки регуляторов (по названию), требуют проверки физического смысла; могут быть полезны модели;
- **ступенчатый** — меняется редко (< 5% часов) и принимает мало значений (например, лабораторные потери q4, q6): интерпретация зависит от процесса и частоты обновления;
- **с оговорками** — часто залипает (одно значение ≥ 24 ч у обычно меняющегося датчика) или даёт одиночные скачки-помехи;
- **годен** — всё в порядке.

Эти категории описывают данные. Ни одна из них не создаёт автоматического исключения из модели.


In [ ]:
Xr = df.loc[running, num_all]
q = pd.DataFrame(index=num_all)
q['пропуски'] = Xr.isna().mean()
q['частое значение'] = Xr.apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 1.0)
q['залипание'] = [float(K.frozen_mask(df[c], running, 24)[running].mean()) for c in num_all]
d1 = df[num_all].where(running).diff(); d2 = df[num_all].where(running).shift(-1) - df[num_all].where(running)
sc_ = 1.4826 * (d1 - d1.median()).abs().median()
q['скачков на 1000 ч'] = (((d1.abs() > 8 * sc_) & (d2.abs() > 8 * sc_) & (np.sign(d1) != np.sign(d2))).sum() / running.sum() * 1000).fillna(0)
q['ступенчатый'] = [K.is_step_like(df[c], running) for c in num_all]
q['норматив'] = [bool(re.search(K.RX_NOT_MEASUREMENT, c, re.I)) for c in num_all]      # нормативы, задания, уставки
def verdict(r):
    if r['частое значение'] >= 0.98: return 'константа'
    if r['пропуски'] > MAX_MISSING: return 'мало данных'
    if r['норматив']: return 'норматив/задание'
    if r['ступенчатый']: return 'ступенчатый'
    if r['залипание'] > 0.10 or r['скачков на 1000 ч'] > 5: return 'с оговорками'
    return 'годен'
q['итог'] = q.apply(verdict, axis=1)
last = df[num_all].apply(lambda s: s.last_valid_index())
# первая и последняя запись датчика — из sensors.csv рядом с часовой таблицей: в самой таблице последнее значение тянется
# до конца данных, и датчик, переставший писаться, выглядит живым (просто постоянным)
s_path = os.path.join(os.path.dirname(HOURLY_PATH), 'sensors.csv')
sinfo = pd.read_csv(s_path, sep=';', encoding='utf-8-sig') if os.path.exists(s_path) else None
if sinfo is not None and {'col', 't_min', 't_max'} <= set(sinfo.columns):
    sinfo['col'] = sinfo['col'].astype(str).str.strip()
    sinfo = sinfo.drop_duplicates('col').set_index('col')
    first = pd.to_datetime(sinfo['t_min'], errors='coerce').reindex(num_all)
    last = pd.to_datetime(sinfo['t_max'], errors='coerce').reindex(num_all)
    src_rec = f'по записям ({s_path})'
else:
    first = df[num_all].apply(lambda s: s.first_valid_index())
    src_rec = 'по часовой таблице (sensors.csv нет — «перестал писаться» может не определиться)'
q['первая запись'], q['последняя запись'] = first, last
early = [c for c in num_all if pd.isna(last[c]) or last[c] < df.index[-1] - pd.Timedelta('30D')]
late = [c for c in num_all if pd.notna(first[c]) and first[c] > df.index[0] + pd.Timedelta('30D') and c not in early]
if late:
    new_tab = pd.DataFrame({'первая запись': first[late], 'пропуски при работе': q.loc[late, 'пропуски'], 'итог': q.loc[late, 'итог']})
    new_tab = new_tab.sort_values('первая запись', ascending=False)
    md(f'**Новые датчики** — появились позже начала данных ({len(late)}):')
    display(K.fmt_df(new_tab.head(20).rename(index=lambda c: short(c, 60)), {'пропуски при работе': '{:.0%}'}))
    new_tab.to_csv(os.path.join(OUT_DIR, 'new_sensors.csv'), encoding='utf-8-sig')
vc = q['итог'].value_counts()
order_v = [v for v in ['годен', 'с оговорками', 'ступенчатый', 'норматив/задание', 'константа', 'мало данных'] if v in vc.index]
colv = {'годен': '#27ae60', 'с оговорками': '#f5b041', 'ступенчатый': '#9b59b6', 'норматив/задание': '#7f8c8d',
        'константа': '#bdc3c7', 'мало данных': '#e74c3c'}
mm = Xr.isna().groupby(Xr.index.to_period('M')).mean()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1, 1.6]})
yy = np.arange(len(order_v))[::-1]
axes[0].barh(yy, [vc[v] for v in order_v], color=[colv[v] for v in order_v])
for y_, v in zip(yy, order_v):
    axes[0].text(vc[v] + 2, y_, str(vc[v]), va='center', fontsize=10)
axes[0].set_yticks(yy); axes[0].set_yticklabels(order_v); axes[0].set_xlabel('датчиков'); axes[0].grid(axis='y', visible=False)
axes[0].set_title(f'Качество {len(num_all)} числовых датчиков')
x_m = [str(p) for p in mm.index]
axes[1].plot(x_m, mm.median(axis=1) * 100, marker='o', color='#2c3e50', label='пропусков у типичного датчика, %')
axes[1].plot(x_m, (mm > 0.5).mean(axis=1) * 100, marker='s', color='#e74c3c', label='датчиков пустых больше чем наполовину, %')
axes[1].set_ylabel('%'); axes[1].tick_params(axis='x', rotation=60); axes[1].legend(loc='upper right')
axes[1].set_title('Пропуски при работе котла по месяцам')
plt.show()
bad = q[q['итог'].ne('годен')].sort_values(['итог', 'пропуски'])
show_cols = ['итог', 'пропуски', 'частое значение', 'залипание', 'скачков на 1000 ч']
display(K.fmt_df(bad[show_cols].head(15).rename(index=lambda c: short(c, 60)),
                 {'пропуски': '{:.0%}', 'частое значение': '{:.0%}', 'залипание': '{:.1%}', 'скачков на 1000 ч': '{:.1f}'}))
q.to_csv(os.path.join(OUT_DIR, 'sensor_quality.csv'), encoding='utf-8-sig')
say('Итог: ' + ', '.join(f'**{v}** — {vc[v]}' for v in order_v) + '.',
    f'Датчики, переставшие давать данные раньше конца периода (> 30 сут; {src_rec}): {len(early)} — {fmt_list([short(c, 30) for c in early], 4)}. '
    'Это диагностическое наблюдение; исключение из модели автоматически не назначается.' if early else '',
    (f'Новых датчиков (первая запись позже начала данных больше чем на 30 сут): {len(late)} — список в `{OUT_DIR}/new_sensors.csv`. '
     'Модель месяца не проверяет датчик, пока в её обучающем окне у него меньше половины значений (`MIN_TRAIN_COVER` в 02): '
     'иначе норма строилась бы по нескольким дням и давала огромные ложные отклонения.') if late else '',
    f'Больше всего пропусков в месяце {mm.mean(axis=1).idxmax()} ({mm.mean(axis=1).max():.0%} в среднем по датчикам).',
    f'Полный список — `{OUT_DIR}/sensor_quality.csv`; в таблице выше — первые 15 отмеченных для проверки.')

## 5. Насколько показания определяются нагрузкой и состоянием механизмов
Так же, как в модели «Аналоги»: для каждого часа установившейся работы находятся 100 самых похожих часов — с той же нагрузкой и тем же состоянием механизмов (текстовые датчики как булевы признаки «механизм не в обычном состоянии»), и по ним предсказывается каждый датчик. Похожие часы берутся только из **других периодов** (данные режутся на 6 кусков по времени), чтобы не подглядывать.

**R² (коэффициент детерминации)** — какая доля изменчивости датчика так объясняется: 1 — полностью, 0 — не лучше, чем «всегда среднее». Считается на данных, которых модель не видела, поэтому бывает **меньше 0**: прогноз по другим периодам хуже простого среднего — значит, уровень датчика в разные периоды разный (дрейф, раздел 6).

In [ ]:
good = [c for c in q.index if c != run_col and c not in MANUAL_EXCLUDE_SENSORS and q.at[c, 'пропуски'] <= MAX_MISSING and df.loc[steady, c].nunique() > 1]  # только для вычислимого R², не список допущенных в модель датчиков
S = steady
Xs = df.loc[S, good].astype(float)
prep = K.Prep().fit(Xs, df.loc[S, txt_cols] if txt_cols else None)
load_med, load_sd = x_run[S].median(), max((x_run[S].quantile(.75) - x_run[S].quantile(.25)) / 1.349, 1e-6)
def conditions(ix):
    R = pd.DataFrame({'нагрузка': ((x_run.loc[ix] - load_med) / load_sd).clip(-10, 10).fillna(0).to_numpy()}, index=ix)
    if prep.bools:
        B = prep._bools(df.loc[ix, txt_cols], prep.bools)
        for j in range(B.shape[1]):
            R[f'b{j}'] = B[:, j]
    return R
R_s = conditions(Xs.index)
med_s, sc_s, z_s, _ = K.analog_crossfit(R_s, Xs, K.time_blocks(Xs.index, 6), 100)
r2 = (1 - ((Xs - med_s) ** 2).mean() / Xs.where((Xs - med_s).notna()).var()).clip(-1, 1)
copies = r2[r2 >= 0.95].index.tolist()                  # почти копии нагрузки — в пример «лучшего» не берём
pick = [(r2.drop(copies) if len(copies) < len(r2) else r2).idxmax(), (r2 - r2.median()).abs().idxmin(), r2.idxmin()]
fig, axes = plt.subplots(1, 4, figsize=(17, 4), gridspec_kw={'width_ratios': [1.3, 1, 1, 1]})
axes[0].hist(r2.dropna(), bins=np.linspace(-1, 1, 41), color='#2e86c1')
axes[0].axvline(0, color='k', lw=1)
axes[0].set_xlabel('R² на данных других периодов (ниже −1 — у −1)'); axes[0].set_ylabel('датчиков')
axes[0].set_title('Насколько условия объясняют датчики')
for ax, c, ttl in zip(axes[1:], pick, ['лучший', 'типичный', 'худший']):
    ax.scatter(med_s[c], Xs[c], s=2, alpha=.25, color='#2c3e50')
    lo, hi = np.nanpercentile(Xs[c], [1, 99]); ax.plot([lo, hi], [lo, hi], color='#c0392b', lw=1)
    ax.set_xlabel('ожидание'); ax.set_ylabel('факт')
    ax.set_title(f'{ttl}, R² = {r2[c]:.2f}\n{lab(c, 38)}', fontsize=9.5)
plt.show()
say(f'Объяснённая доля изменчивости: медиана R² = **{r2.median():.2f}**; хорошо объясняются (R² > 0.5) {int((r2 > .5).sum())} из {len(good)} датчиков, '
    f'почти не объясняются (R² < 0.2) — {int((r2 < .2).sum())}.',
    (f'R² ≥ 0.95 у {len(copies)} — это почти копии нагрузки ({fmt_list([short(c, 30) for c in copies[:3]])}); '
     'как пример «лучшего» взят следующий за ними.') if copies else '',
    f'Состояний механизмов, взятых как условия (булевы признаки): {len(prep.bools)} из {len(txt_cols)} текстовых — остальные почти не меняются или дублируют друг друга.',
    f'R² < 0 у {int((r2 < 0).sum())} датчиков: их уровень в разные периоды разный (дрейф), см. раздел 6.' if (r2 < 0).any() else '',
    'На точечных графиках: точки на красной диагонали — прогноз точен. Низкий R² не значит «плохой датчик»: у химконтроля, подшипников и т.п. '
    'изменения идут не от нагрузки, и для них норма — их собственный обычный уровень.')

# отклонения z для всех часов работы (понадобятся в разделах 6–10)
am = K.AnalogModel(100).fit(R_s, Xs)
Z = pd.DataFrame(np.nan, index=df.index, columns=good)
Z.loc[S] = z_s.to_numpy()
other = running & ~S
if other.any():
    Z.loc[other] = am.expected(conditions(df.index[other]), df.loc[other, good].astype(float))[2].to_numpy()


## 6. Стабильна ли норма во времени
Для каждого датчика — насколько его обычное отклонение от ожидания (раздел 5) в каждом месяце сдвинуто от обычного за весь период, в единицах его разброса. Сдвиг больше 1 — датчик в этом месяце заметно «уплыл»: износ, загрязнение, перекалибровка или изменения после ремонта.

In [ ]:
resid = Xs - med_s
iqr_r = (resid.quantile(.75) - resid.quantile(.25)).replace(0, np.nan)
per = resid.index.to_period('M')
cnt = resid.notna().groupby(per).sum()
shift = ((resid.groupby(per).median() - resid.median()) / iqr_r).where(cnt >= 72)
share_m = (shift.abs() > 1).mean(axis=1).where(shift.notna().any(axis=1))
share_m = share_m.reindex(pd.period_range(per.min(), per.max(), freq='M'))   # месяцы без работы — пустые, но на оси
drift = shift.abs().max().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.3, 1]})
axes[0].bar([str(p) for p in share_m.index], share_m * 100, color='#2e86c1')
axes[0].set_ylabel('% датчиков'); axes[0].tick_params(axis='x', rotation=60)
for i in np.flatnonzero(share_m.isna().to_numpy()):
    axes[0].text(i, 0, 'нет\nработы', ha='center', va='bottom', fontsize=7, color='#95a5a6')
axes[0].set_title('Доля датчиков, «уплывших» больше чем на 1 разброс, по месяцам')
top = drift.head(10)[::-1]
axes[1].barh(np.arange(len(top)), top.to_numpy(), color='#e67e22')
axes[1].set_yticks(np.arange(len(top))); axes[1].set_yticklabels([lab(c, 50) for c in top.index], fontsize=9)
axes[1].set_xlabel('наибольший сдвиг за месяц, в разбросах'); axes[1].grid(axis='y', visible=False)
axes[1].set_title('Сильнее всего дрейфуют')
plt.show()
say(f'Хотя бы в одном месяце «уплывают» больше чем на 1 разброс **{int((drift > 1).sum())} из {len(good)}** датчиков; в типичный месяц — {pct(share_m.dropna().median())}.',
    f'Сильнее всего: {fmt_list([f"{short(c, 30)} ({v:.1f})" for c, v in drift.head(5).items()])}.',
    '**Вывод:** норма не постоянна — модель нужно переобучать на свежих данных (в ноутбуке моделей — каждый месяц на последнем годе).')

## 7. Почему пороги берутся «по факту», а не по правилу 3σ
Многие стандартные методы молча предполагают, что отклонения распределены «нормально» (колокол Гаусса): тогда отклонение больше 3σ бывает лишь в 0.27% случаев, и порог «3σ» даёт мало ложных тревог. Проверяем на отклонениях z из раздела 5.

In [ ]:
za = z_s.abs().to_numpy().ravel(); za = za[np.isfinite(za)]
tab7 = pd.DataFrame({'в норме (факт)': {t: float(np.mean(za > t)) for t in (3, 5, 10)},
                     'по закону Гаусса': {t: 2 * stats.norm.sf(t) for t in (3, 5, 10)}})
tab7['во сколько раз чаще'] = tab7['в норме (факт)'] / tab7['по закону Гаусса']
tab7.index = [f'|z| > {t}' for t in tab7.index]
display(K.fmt_df(tab7, {'в норме (факт)': '{:.2%}', 'по закону Гаусса': '{:.1e}', 'во сколько раз чаще': '{:,.0f}'}))
tail3 = (z_s.abs() > 3).mean().sort_values()
ex = [tail3.index[len(tail3) // 2], tail3.index[-1]]
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
gx = np.linspace(-8, 8, 400)
for ax, c, ttl in zip(axes, ex, ['типичный датчик', 'самый «хвостатый»']):
    ax.hist(z_s[c].dropna().clip(-8, 8), bins=np.linspace(-8, 8, 81), density=True, color='#27ae60', label='факт в норме')
    ax.plot(gx, stats.norm.pdf(gx), 'k--', lw=1, label='колокол Гаусса')
    ax.set_yscale('log'); ax.set_ylim(1e-5, 1); ax.set_xlabel('отклонение z'); ax.legend(loc='upper right')
    ax.set_title(f'{ttl}: {short(c, 40)} — |z| > 3 в {tail3[c]:.1%} часов', fontsize=10)
plt.show()
say(f'|z| > 3 в нормальной работе встречается в **{pct(tab7.iloc[0, 0], 2)}** значений (по Гауссу — 0.27%), |z| > 5 — в {pct(tab7.iloc[1, 0], 2)} '
    f'(в {tab7.iloc[1, 2]:,.0f} раз чаще, чем по Гауссу).',
    '**Вывод:** порог «3σ» тревожил бы постоянно. Поэтому в моделях порог — по реальной частоте в норме: «такое бывало лишь в 0.5% часов нормы».')

## 8. Дубли датчиков и «память» отклонений
- **Дубли («близнецы»)** — датчики, которые в норме почти повторяют друг друга (ранговая корреляция ≥ 0.95): левый/правый поток, соседние термопары, два прибора на одну величину. Расхождение дублей — неисправность датчика или перекос процесса.
- **Память отклонений** — если датчик отклонился, как долго это держится: корреляция отклонения z с самим собой через 1 ч, сутки, неделю.

In [ ]:
Sp = Xs.iloc[::2].rank().corr().fillna(0)
cl = pd.Series(hierarchy.fcluster(K.corr_linkage(Sp, 'complete'), 0.05, 'distance'), index=Sp.index)
twin_groups = sorted([g.index.tolist() for _, g in cl.groupby(cl) if len(g) > 1], key=len, reverse=True)
if twin_groups:
    display(pd.DataFrame({'датчиков': [len(g) for g in twin_groups[:12]],
                          'состав': [fmt_list([short(c, 35) for c in g], 3) for g in twin_groups[:12]]}))
Zs = Z.where(S)
acf = {L_: float(Zs.corrwith(Zs.shift(L_)).median()) for L_ in (1, 24, 168)}
say(f'Групп дублей: **{len(twin_groups)}**, в них {sum(map(len, twin_groups))} датчиков (выше — первые 12 групп). В ноутбуке моделей их расхождение '
    'проверяет метод «Близнецы», а в остальных методах из почти одинаковых остаётся один — чтобы одна величина не «голосовала» несколько раз.',
    f'Память отклонений: корреляция через 1 ч — {acf[1]:.2f}, через сутки — {acf[24]:.2f}, через неделю — {acf[168]:.2f}. '
    'Отклонения держатся долго, поэтому один странный час — не повод для тревоги: в моделях ошибка усредняется за несколько часов и тревога требует устойчивости.')

## 9. Текстовые датчики: задвижки, механизмы, защиты
Сколько текстовых датчиков реально меняются (они становятся условиями работы в моделях), а сколько почти всегда в одном состоянии. Отдельно — подозрительные значения: тайм-ауты, самоход, срабатывание защит.

In [ ]:
SUSP = K.RX_SUSPICIOUS_TEXT                      # тайм-ауты, самоход, срабатывание защит (k2_lib.py)
if txt_cols:
    T = df.loc[running, txt_cols]
    changing = [c for c in txt_cols if 1 - T[c].value_counts(normalize=True).iloc[0] >= 0.02] if len(T) else []
    rows = []
    for c in txt_cols:
        vv = T[c].dropna()
        fq = df.loc[steady, c].value_counts(normalize=True)
        for val, n in vv[vv.astype(str).str.lower().str.contains(SUSP, regex=True)].value_counts().items():
            if fq.get(val, 0) < 0.05:
                rows.append({'датчик': short(c, 55), 'значение': val, 'часов при работе': int(n), 'доля в норме': fq.get(val, 0)})
    susp = pd.DataFrame(rows)
    if len(susp):
        display(K.fmt_df(susp.sort_values('часов при работе', ascending=False).head(15), {'доля в норме': '{:.2%}'}))
    say(f'Текстовых датчиков: {len(txt_cols)}. Меняются хотя бы 2% времени работы: **{len(changing)}**; после удаления дублей в модели как условия '
        f'работы идут **{len(prep.bools)}**. Остальные почти всегда в одном состоянии — их редкие значения ловит метод «Редкие состояния».',
        f'Подозрительных значений (тайм-аут, самоход, срабатывание и т.п.), редких в норме: {len(susp)} вариантов — таблица выше.' if len(susp) else
        'Подозрительных значений при работе котла не найдено.')
else:
    say('Текстовых датчиков нет.')

## 10. Какие датчики отклонялись перед ремонтами
Для каждого ремонта — последние `PRE_HOURS` ч работы котла перед ним и датчики, которые отклонялись от ожидания (раздел 5) сильнее всего. Отклонение датчика делится на его обычный уровень в норме (95-й процентиль): **1 = верхняя граница нормы**, 2 = вдвое дальше. Берётся 90-й процентиль за период — насколько далеко датчик уходил в самые необычные часы.

Резервы и плановые ремонты — для сравнения: котёл перед ними был исправен, поэтому датчики, «горящие» и там, скорее связаны с подготовкой к останову, а не с неисправностью. Это описание, а не доказательство: аварий мало, и причины у них разные.

In [ ]:
days_to_fail = K.days_to_next(df.index, fails.start)
typ = Z.abs().where(steady & (days_to_fail > 14)).quantile(0.95).clip(lower=1)
rel = Z.abs() / typ
panels, rows = [], []
for e in pd.concat([fails, reserves, plans]).itertuples():
    w = df.index[(df.index < e.start) & running.to_numpy() & (df.index >= e.start - pd.Timedelta(days=14))][-PRE_HOURS:]
    if len(w) < 12:
        rows.append({'ремонт': f'{EV_NAME[e.type]} {e.start:%Y-%m-%d}', 'часов работы перед ним': len(w), 'главные датчики': 'котёл перед ремонтом не работал'})
        continue
    sh = rel.loc[w].quantile(0.9).dropna().sort_values(ascending=False)
    panels.append((e, sh.head(8)))
    rows.append({'ремонт': f'{EV_NAME[e.type]} {e.start:%Y-%m-%d}', 'часов работы перед ним': len(w),
                 'главные датчики': ', '.join(f'{short(c, 35)} ×{v:.1f}' for c, v in sh.head(3).items())})
ncol = 3
nrow = int(np.ceil(len(panels) / ncol)) if panels else 0
if nrow:
    fig, axes = plt.subplots(nrow, ncol, figsize=(17, 3.9 * nrow), squeeze=False)
    for ax, (e, sh) in zip(axes.ravel(), panels):
        s_ = sh[::-1]
        ax.barh(np.arange(len(s_)), s_.to_numpy(), color=EV_COL[e.type])
        ax.set_yticks(np.arange(len(s_))); ax.set_yticklabels([lab(c, 46) for c in s_.index], fontsize=8.5)
        ax.axvline(1, color='gray', ls='--', lw=1)
        ax.set_title(f'{EV_NAME[e.type].capitalize()} {e.start:%Y-%m-%d}', fontsize=11, color=EV_COL[e.type])
        ax.set_xlabel('отклонение / обычный уровень'); ax.grid(axis='y', visible=False)
    for ax in axes.ravel()[len(panels):]:
        ax.axis('off')
    fig.suptitle(f'Датчики, сильнее всего отклонявшиеся в последние {PRE_HOURS} ч работы перед ремонтом (пунктир — граница нормы)',
                 fontsize=12, fontweight='bold')
    plt.show()
display(pd.DataFrame(rows))
say('Красные — аварии, оранжевые — резервы, синие — плановые ремонты.',
    'Датчики, которые выходят за норму и перед авариями, и перед резервами/плановыми, скорее реагируют на подготовку к останову. '
    'Датчики только перед авариями — кандидаты в предвестники; их стоит сверить с содержанием заявок (раздел 2) и показать технологам.')

### 10.1 Небаланс вода − пар
Расход питательной воды должен почти совпадать с расходом пара (разница — продувка и подпитка). Если разница растёт —
вода уходит не туда: свищ экранной трубы, течь в тракте, или врёт расходомер. Это видно без всякой модели. Небаланс — в
процентах нагрузки, отклонение — от его обычного уровня (медиана в нормальной работе). Эпизод — отклонение больше
`IMB_PCT` % нагрузки дольше `IMB_HOURS` ч подряд при работе котла. Датчики — по шаблонам `RX_FEED_WATER_FLOW` и `RX_STEAM_FLOW`. Для каждого эпизода — что было в ближайшие 14 сут после
него: авария, плановый ремонт, резерв или останов без заявки. В новом проекте такие эпизоды не идут в обучение и дают
тревогу «выше допустимой границы»; здесь — только описание.

In [ ]:
feed_c = [c for c in num_all if K.name_hit(RX_FEED_WATER_FLOW, c)]
feed_col = min(feed_c, key=lambda c: df.loc[NORMAL02, c].isna().mean()) if feed_c else None
steam_c = [c for c in [run_col] + num_all if K.name_hit(RX_STEAM_FLOW, c)]          # сначала — датчик работы, если это расход пара
steam_col = steam_c[0] if steam_c else None
IMB_EP = pd.DataFrame()
if feed_col is None or steam_col is None:
    say('Небаланс не посчитать: не найден ' + ('расход питательной воды' if feed_col is None else 'расход пара')
        + ' (шаблоны RX_FEED_WATER_FLOW / RX_STEAM_FLOW в k2_lib.py; сокращение «F …» понимается).')
else:
    ok_ = running & mode.eq(M_RUN) & (df[steam_col].astype(float) > run_thr)
    imb = ((df[feed_col].astype(float) - df[steam_col].astype(float)) / df[steam_col].astype(float) * 100).where(ok_)
    base_imb = float(imb[NORMAL02].median())
    dev_ = (imb - base_imb).abs()
    flag_ = (dev_ > IMB_PCT).fillna(False)
    ep_ = K.episodes(flag_)
    ep_ = ep_[ep_.value.eq(True) & ep_.hours.ge(IMB_HOURS)]
    stops_nt = [r.start for r in stop_ep.itertuples() if not ((events.start < r.end) & (events.end > r.start)).any()]
    nxt_all = pd.concat([events[['start', 'type']].assign(вид=lambda d_: d_['type'].map(EV_NAME)),
                         pd.DataFrame({'start': stops_nt, 'вид': 'останов без заявки'})], ignore_index=True).sort_values('start')
    rows_i = []
    for r in ep_.itertuples():
        nx = nxt_all[(nxt_all['start'] >= r.start) & (nxt_all['start'] <= r.end + pd.Timedelta('14D'))]
        rows_i.append({'начало': r.start, 'конец': r.end, 'часов': int(r.hours),
                       'небаланс, % нагрузки (макс.)': float(imb[r.start:r.end].abs().max()),
                       'в ближайшие 14 сут': f'{nx["вид"].iloc[0]} {nx["start"].iloc[0]:%Y-%m-%d}' if len(nx) else 'ничего'})
    IMB_EP = pd.DataFrame(rows_i)
    fig, ax = plt.subplots(figsize=(15, 3.8))
    shade(ax)
    d_ = imb.resample('1D').mean()
    ax.plot(d_.index, d_, color='#2c3e50', lw=1)
    ax.axhspan(base_imb - IMB_PCT, base_imb + IMB_PCT, color='#27ae60', alpha=.12, lw=0)
    for r in ep_.itertuples():
        ax.axvspan(r.start, r.end, ymin=0.92, ymax=1.0, color='#8e44ad', lw=0)
    ax.set_ylabel('вода − пар, % нагрузки'); K.date_axis(ax)
    ax.set_title(f'Небаланс «{short(feed_col, 30)}» − «{short(steam_col, 30)}» (среднее за сутки); зелёное — обычный ± {IMB_PCT:g}%, '
                 'фиолетовое — эпизоды', fontsize=10)
    plt.show()
    if len(IMB_EP):
        display(K.fmt_df(IMB_EP, {'небаланс, % нагрузки (макс.)': '{:.1f}'}))
        IMB_EP.to_csv(os.path.join(OUT_DIR, 'imbalance_episodes.csv'), index=False, encoding='utf-8-sig')
    n_fail_ = int(IMB_EP['в ближайшие 14 сут'].str.startswith(EV_NAME[EV_FAIL]).sum()) if len(IMB_EP) else 0
    say(f'Обычный небаланс — {base_imb:+.1f}% нагрузки. Эпизодов (отклонение > {IMB_PCT:g}% дольше {IMB_HOURS} ч): **{len(IMB_EP)}**'
        + (f', из них перед аварией (≤ 14 сут): **{n_fail_}**' if len(IMB_EP) else '') + '.',
        'Эпизоды перед авариями и остановами без заявки — кандидаты в «течь, которую видно заранее»; эпизоды, после которых '
        'ничего не было, — скорее расходомеры. Таблица — `imbalance_episodes.csv`.' if len(IMB_EP) else '')

## 11. Итоги и решения для модели

In [ ]:
decisions = {
    'created': pd.Timestamp.now().isoformat(timespec='seconds'),
    'run_col': run_col, 'run_thr': float(run_thr), 'startup_h': int(startup_h),
    'first_test': K.first_test_month(df.index, steady & ~pd.Series(after_rep, index=df.index), MIN_NORMAL_H),   # как норма в 02
    'normal_hours': int(NORMAL02.sum()), 'months_without_training': short_m,          # раздел 3.2
    'label_offset_h': LABEL_OFFSET_H,                                                  # раздел 3.1: заявка позже останова, ч (медиана)
    'conditions_found': COND_FOUND, 'feed_col': feed_col, 'steam_col': steam_col,
    'imbalance_episodes': len(IMB_EP),
    **K.sensor_review_decisions(q, MANUAL_EXCLUDE_SENSORS),
    'sensors_ended_early': [],  # без автоматического запрета по концу всего файла
    'sensors_ended_early_review': early,
    'twin_groups': twin_groups,
    'run_col_source': run_src,
}
with open(os.path.join(OUT_DIR, 'eda_decisions.json'), 'w', encoding='utf-8') as f:
    json.dump(decisions, f, ensure_ascii=False, indent=1, default=str)
say(f'**Данные.** {days:.0f} сут, {len(num_all)} числовых и {len(txt_cols)} текстовых датчиков; аварий {len(fails)}, плановых ремонтов {len(plans)}, '
    f'выводов в резерв {len(reserves)} (метки — по типу и категории заявки).',
    f'**Работа котла** — по «{short(run_col, 50)}» ({run_src}) > {run_thr:.3g}; пуск ~{startup_h} ч; норма для моделей — установившаяся работа ({steady.sum():,} ч).',
    f'**Время заявок:** {verdict_off}',
    f'**Норма для обучения** (как в 02, без {EXCL_FAIL_D} сут перед авариями): {int(NORMAL02.sum()):,} ч'
    + (f'; месяцы без обучения (< {MIN_TRAIN_H} ч): {fmt_list(short_m, 8)}' if short_m else '') + '. '
    + 'Условия для 02: ' + ', '.join(f'{k} — {"есть" if v else "НЕ НАЙДЕНА"}' for k, v in COND_FOUND.items()) + '.',
    (f'**Небаланс вода − пар:** эпизодов {len(IMB_EP)}' + (f', перед авариями — {int(IMB_EP["в ближайшие 14 сут"].str.startswith(EV_NAME[EV_FAIL]).sum())}' if len(IMB_EP) else '')
     + '.') if feed_col and steam_col else '**Небаланс вода − пар** не посчитан (нет расхода питательной воды или пара).',
    (f'**Проверка моделей** начнётся с {decisions["first_test"]}: до этого месяца набирается {MIN_NORMAL_H} ч нормальной работы, на которых учится первая модель.'
     if decisions['first_test'] else f'**Нормальной работы меньше {MIN_NORMAL_H} ч** — проверять модели не на чем.'),
    f'**Датчики.** Годны {int(vc.get("годен", 0))}, с оговорками {int(vc.get("с оговорками", 0))}, требуют проверки '
    f'{int(sum(vc.get(k, 0) for k in ["константа", "мало данных", "ступенчатый", "норматив/задание"]))}.',
    (f'**Норма зависит от условий:** нагрузка и состояние механизмов объясняют медианно R² = {r2.median():.2f} изменчивости датчиков — '
     'поэтому сравнивать надо с похожими по условиям часами (метод аналогов).') if r2.median() >= 0.2 else
    (f'**Нагрузка и состояние механизмов объясняют немного** (медиана R² = {r2.median():.2f}): у многих датчиков изменения идут от того, чего '
     'нет в условиях (сезон, дрейф, оборудование без текстовых состояний, например мельницы). Условия работы для аналогов стоит расширить.'),
    f'**Норма дрейфует:** {int((drift > 1).sum())} датчиков хотя бы в одном месяце «уплывают» больше чем на разброс — нужно ежемесячное переобучение.',
    f'**Хвосты тяжёлые:** |z| > 5 в {pct(tab7.iloc[1, 0], 2)} часов нормы — пороги только по реальной частоте в норме.',
    f'**Отклонения держатся долго** (корреляция через сутки {acf[24]:.2f}) — тревога должна требовать устойчивости.',
    f'Решения сохранены в `{OUT_DIR}/eda_decisions.json`, качество датчиков — в `{OUT_DIR}/sensor_quality.csv`.')